# Differentiable Monte Carlo — The Real Deal

**Goal:** Convert the paper's MC simulation into a fully differentiable pipeline in PyTorch.

---

## Step 1: Imports & Setup

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import dataclass
import math

## Step 2: Parameters & Sampling Functions

In [ ]:
@dataclass
class MCParams:
    gamma: float       # HWHM of Cauchy (MHz) — optimized
    nbar: float        # mean photon count — optimized
    sigma: float = 6.0  # noise std (fixed, from paper)
    lambda_: float = 2.0  # mean background counts (fixed, from paper)

# Frequency window (from paper)
FREQ_MIN = -75.0   # MHz
FREQ_MAX = 75.0    # MHz

def sample_n(params, epsilon):
    """Sample photon count N ~ N(nbar, sigma), reparameterized."""
    n_float = params.nbar + params.sigma * epsilon
    return max(round(n_float), 0)

def sample_cauchy(n, gamma):
    """Sample n photon frequencies from Cauchy(gamma)."""
    if n == 0:
        return torch.tensor([], dtype=torch.float32)
    u = np.random.uniform(0, 1, n)
    samples = gamma * np.tan(np.pi * (u - 0.5))
    return torch.tensor(np.clip(samples, FREQ_MIN, FREQ_MAX), dtype=torch.float32)

def sample_background(lambda_):
    """Sample background events uniformly across freq window."""
    n_bg = np.random.poisson(lambda_)
    if n_bg == 0:
        return torch.tensor([], dtype=torch.float32)
    bg = np.random.uniform(FREQ_MIN, FREQ_MAX, n_bg)
    return torch.tensor(bg, dtype=torch.float32)

def sample_photons(n, gamma, lambda_):
    """Combine Cauchy signal + background into one photon tensor."""
    signal = sample_cauchy(n, gamma)
    bg = sample_background(lambda_)
    return torch.cat([signal, bg])

## Step 3: Fitting — Voigt MLE

In [ ]:
def pseudo_voigt_log_pdf(freqs, center, log_gamma, log_sigma_g, logit_eta):
    """Log-PDF of pseudo-Voigt. Params in unconstrained space for stable opt."""
    gamma = torch.exp(log_gamma)
    sigma_g = torch.exp(log_sigma_g)
    eta = torch.sigmoid(logit_eta)
    
    gauss = torch.exp(-0.5 * ((freqs - center) / sigma_g) ** 2)
    gauss = gauss / (sigma_g * torch.sqrt(torch.tensor(2.0 * torch.pi)))
    
    lorentz = (gamma / torch.pi) / ((freqs - center) ** 2 + gamma ** 2)
    
    pdf = eta * gauss + (1 - eta) * lorentz
    return torch.log(pdf + 1e-30)

def fit_pseudo_voigt(photons, n_iters=200):
    """
    Fit pseudo-Voigt to photon frequencies via MLE (L-BFGS).
    Returns (fwhm, params_dict).
    """
    if len(photons) < 3:
        return 50.0, None
    
    freqs = photons.clone().detach().float()
    
    center = torch.tensor(float(freqs.median()), requires_grad=True)
    log_gamma = torch.tensor(np.log(15.0), requires_grad=True)
    log_sigma_g = torch.tensor(np.log(5.0), requires_grad=True)
    logit_eta = torch.tensor(0.0, requires_grad=True)
    
    optimizer = torch.optim.LBFGS([center, log_gamma, log_sigma_g, logit_eta],
                                   max_iter=n_iters, line_search_fn='strong_wolfe')
    
    def closure():
        optimizer.zero_grad()
        log_pdf = pseudo_voigt_log_pdf(freqs, center, log_gamma, log_sigma_g, logit_eta)
        nll = -log_pdf.mean()
        nll.backward()
        return nll
    
    try:
        optimizer.step(closure)
    except RuntimeError:
        return 50.0, None
    
    gamma_val = torch.exp(log_gamma).item()
    if not math.isfinite(gamma_val):
        return 50.0, None
    fwhm = 2.0 * gamma_val
    
    return fwhm, {
        'center': center.item(),
        'gamma': gamma_val,
        'sigma_g': torch.exp(log_sigma_g).item(),
        'eta': torch.sigmoid(logit_eta).item(),
        'fwhm': fwhm,
    }

## Step 4: Full Run — Params In, FWHM Out

In [ ]:
def full_run(params, epsilon):
    """One full MC run: sample N → Cauchy photons → background → fit → FWHM."""
    n = sample_n(params, epsilon)
    photons = sample_photons(n, params.gamma, params.lambda_)
    if len(photons) < 3:
        return 50.0  # too few photons, return safe default
    
    fwhm, _ = fit_pseudo_voigt(photons)
    return fwhm

## Step 5: Full Simulation

In [ ]:
def simulate(params, n_runs=2000, seed=None):
    """Run N MC runs with the same params. Returns tensor of FWHMs."""
    if seed is not None:
        np.random.seed(seed)
    
    fwhms = torch.zeros(n_runs)
    for i in range(n_runs):
        eps = np.random.normal()
        fwhms[i] = full_run(params, eps)
    
    return fwhms

## Step 6: Kernel Density Estimate

In [ ]:
def kde(fwhms, x_grid, bandwidth=2.0):
    """
    1D Gaussian KDE. Fully differentiable.
    
    Args:
        fwhms: (N,) tensor — extracted linewidths from simulate()
        x_grid: (M,) tensor — positions to evaluate density
        bandwidth: float — smoothing width (FWHM units, MHz)
    
    Returns:
        density: (M,) tensor — smooth density evaluated on x_grid
    """
    if len(fwhms) == 0:
        return torch.zeros_like(x_grid)
    
    diff = fwhms[:, None] - x_grid[None, :]
    kernel = torch.exp(-0.5 * (diff / bandwidth) ** 2)
    kernel = kernel / (bandwidth * math.sqrt(2.0 * math.pi))
    return kernel.mean(dim=0)

---
Functions loaded. Ready for testing.

## Step 7: Loss Function — KDE to Smooth Bin Counts + L2

Two separate functions:
1. `kde_to_bin_counts` — integrates the KDE over each experimental bin using erf (differentiable)
2. `l2_loss` — compares two histograms via L2 on probability densities

In [ ]:
def kde_to_bin_counts(fwhms, bin_edges, bandwidth=2.0):
    """
    Convert FWHM point cloud to smooth bin counts via KDE integration.
    
    Each FWHM is a Gaussian kernel. erf gives us the exact integral
    over each bin interval. Fully differentiable.
    
    Args:
        fwhms: (N,) tensor — extracted linewidths from simulate()
        bin_edges: (M+1,) tensor — bin boundaries of experimental histogram
        bandwidth: float — KDE bandwidth (MHz)
    
    Returns:
        counts: (M,) tensor — smooth bin counts (can be fractional)
    """
    left = bin_edges[:-1]   # (M,)
    right = bin_edges[1:]  # (M,)
    
    s = bandwidth * math.sqrt(2.0)
    erf_right = torch.erf((right[None, :] - fwhms[:, None]) / s)
    erf_left  = torch.erf((left[None, :]  - fwhms[:, None]) / s)
    
    # (N, M) — fraction of each kernel's mass in each bin
    bin_probs = 0.5 * (erf_right - erf_left)
    
    # (M,) — sum over all kernels = smooth bin counts
    return bin_probs.sum(dim=0)

def l2_loss(sim_counts, exp_counts):
    """
    L2 distance on probability densities.
    Both inputs are bin counts (normalized internally).
    
    Args:
        sim_counts: (M,) tensor — simulated bin counts (from kde_to_bin_counts)
        exp_counts: (M,) tensor — experimental bin counts
    
    Returns:
        loss: scalar
    """
    sim_pdf = sim_counts / sim_counts.sum()
    exp_pdf = exp_counts / exp_counts.sum()
    return ((sim_pdf - exp_pdf) ** 2).sum()


# Test: generate dummy experimental data, recover params
print("--- Testing loss function ---")

# "Real" params
true_params = MCParams(gamma=20.0, nbar=50.0)
np.random.seed(123)
real_fwhms = simulate(true_params, n_runs=500, seed=123)

# Build experimental bins from the real data (simulate what we'd get from experiment)
exp_bin_edges = torch.linspace(0, 100, 25)
exp_counts = torch.histc(real_fwhms, bins=len(exp_bin_edges)-1, min=0, max=100)

# Simulate with different params, compute loss
guess_params = MCParams(gamma=10.0, nbar=40.0)
sim_fwhms = simulate(guess_params, n_runs=500, seed=456)

sim_counts = kde_to_bin_counts(sim_fwhms, exp_bin_edges, bandwidth=3.0)
loss = l2_loss(sim_counts, exp_counts)

print(f'Real:     gamma={true_params.gamma}, nbar={true_params.nbar}')
print(f'Guess:    gamma={guess_params.gamma}, nbar={guess_params.nbar}')
print(f'L2 Loss:  {loss:.4f}')

# Verify: same params should give lower loss
same_params = MCParams(gamma=20.0, nbar=50.0)
same_fwhms = simulate(same_params, n_runs=500, seed=789)
same_counts = kde_to_bin_counts(same_fwhms, exp_bin_edges, bandwidth=3.0)
same_loss = l2_loss(same_counts, exp_counts)
print(f'Same params loss: {same_loss:.4f} (should be lower)')